### 1. Imports and data loading

In [ ]:
import pandas as pd
import numpy as np

from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [175]:
# =========================
# STEP 1: Load data
# =========================

data_path = "/Users/ramilmammadov/Desktop/Capstone Project/GitHub Repo/Capstone_Citizensbank_climate-risk_insurance-exposure_monitoring/02_processed_data/wildfire_drought_hpi_flood_heat_huricane.csv"
df = pd.read_csv(data_path)

print("Shape:", df.shape)
print(df.head())

print("\nMissing values per column:")
print(df.isna().sum())

print("\nYear range:", df["yr"].min(), "to", df["yr"].max())

Shape: (21965, 17)
     yr   hpi_yoy       hpi  STATE   CZ_NAME  FLOOD_FREQUENCY  \
0  2000  1.226954  115.7100      0         0              0.0   
1  2001  4.070521  120.4200      0         0              0.0   
2  2002  2.850440  123.8525  TEXAS  CALLAHAN              2.0   
3  2003  2.620052  127.0975  TEXAS  CALLAHAN              2.0   
4  2004  4.455241  132.7600  TEXAS  CALLAHAN              4.0   

   FLOOD_PROPERTY_DAMAGE  FLOOD_DURATION_HOURS  HEAT_VALUE  HEAT_INDEX  \
0                    0.0                  0.00        94.4    0.289948   
1                    0.0                  0.00        95.7    0.353297   
2               807000.0                 14.08        90.8    0.059089   
3                    0.0                  2.00        92.8    0.178862   
4               100000.0                  4.18        89.6   -0.000561   

        HSI  DROUGHT_ANNUAL_MEAN_INDEX  FIRE_FREQUENCY  FIRE_SIZE  \
0  0.509057                      -3.36             0.0        0.0   
1  0.58

In [ ]:
# =========================
# STEP 2: Restrict to 2000–2017 for modeling
# =========================

TARGET_COL = "hpi_yoy"

df_model = df[df["yr"].between(2000, 2017)].copy()

print("\nModeling data years:", df_model["yr"].min(), "to", df_model["yr"].max())
print("Number of rows in modeling data:", len(df_model))


Modeling data years: 2000 to 2017
Number of rows in modeling data: 18827


In [ ]:
# =========================
# STEP 3: Create lag features per (STATE, CZ_NAME)
# =========================

group_cols = ["STATE", "CZ_NAME"]

df_model = df_model.sort_values(group_cols + ["yr"])

grp = df_model.groupby(group_cols)["hpi_yoy"]

df_model["hpi_yoy_lag1"] = grp.shift(1)
df_model["hpi_yoy_roll3"] = grp.shift(1).rolling(window=3, min_periods=1).mean()

df_model = df_model.dropna(subset=["hpi_yoy_lag1"]).copy()

In [ ]:
# =========================
# STEP 4: Define feature columns (numeric + categorical)
# =========================

all_cols = df_model.columns.tolist()

feature_cols = [c for c in all_cols if c != TARGET_COL]

categorical_cols = ["STATE", "CZ_NAME"]
numeric_cols = [c for c in feature_cols if c not in categorical_cols]

print("\nNumeric features:", numeric_cols)
print("Categorical features:", categorical_cols)


Numeric features: ['yr', 'hpi', 'FLOOD_FREQUENCY', 'FLOOD_PROPERTY_DAMAGE', 'FLOOD_DURATION_HOURS', 'HEAT_VALUE', 'HEAT_INDEX', 'HSI', 'DROUGHT_ANNUAL_MEAN_INDEX', 'FIRE_FREQUENCY', 'FIRE_SIZE', 'deaths_hurricane', 'injuries_hurricane', 'damage_hurricane', 'hpi_yoy_lag1', 'hpi_yoy_roll3']
Categorical features: ['STATE', 'CZ_NAME']


In [ ]:
# =========================
# STEP 5: Time-based splits: train / validation / test
# =========================

train_mask = df_model["yr"].between(2000, 2009)
val_mask = df_model["yr"].between(2010, 2012)
test_mask = df_model["yr"].between(2013, 2017)


def split_xy(mask):
    df_part = df_model[mask].copy()
    X_num = df_part[numeric_cols]
    X_cat = df_part[categorical_cols]
    y = df_part[TARGET_COL]
    return X_num, X_cat, y


X_train_num, X_train_cat, y_train = split_xy(train_mask)
X_val_num, X_val_cat, y_val = split_xy(val_mask)
X_test_num, X_test_cat, y_test = split_xy(test_mask)

print("\nTrain size:", len(y_train))
print("Validation size:", len(y_val))
print("Test size (not used yet):", len(y_test))


Train size: 9356
Validation size: 3116
Test size (not used yet): 5207


In [ ]:
# =========================
# STEP 6: One-hot encode categoricals & build final X matrices
# =========================

# One-hot encode training categorical features
X_train_cat_dum = pd.get_dummies(X_train_cat, drop_first=False)

# Apply to validation + test
X_val_cat_dum = pd.get_dummies(X_val_cat)
X_test_cat_dum = pd.get_dummies(X_test_cat)

# Align columns to training dummies
X_val_cat_dum = X_val_cat_dum.reindex(columns=X_train_cat_dum.columns, fill_value=0)
X_test_cat_dum = X_test_cat_dum.reindex(columns=X_train_cat_dum.columns, fill_value=0)

# Combine numeric + categorical dummy features
X_train = pd.concat(
    [X_train_num.reset_index(drop=True), X_train_cat_dum.reset_index(drop=True)], axis=1
)

X_val = pd.concat(
    [X_val_num.reset_index(drop=True), X_val_cat_dum.reset_index(drop=True)], axis=1
)

X_test = pd.concat(  # prepared for later, not evaluated yet
    [X_test_num.reset_index(drop=True), X_test_cat_dum.reset_index(drop=True)], axis=1
)

print("\nFinal feature shapes:")
print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test (for later):", X_test.shape)


Final feature shapes:
Train: (9356, 828)
Validation: (3116, 828)
Test (for later): (5207, 828)


In [ ]:
# from xgboost import XGBRegressor
# from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
# import numpy as np

# def regression_metrics(y_true, y_pred):
#     mse = mean_squared_error(y_true, y_pred)
#     rmse = mse ** 0.5
#     mae  = mean_absolute_error(y_true, y_pred)
#     r2   = r2_score(y_true, y_pred)
#     return rmse, mae, r2

# def time_series_r2(y_train, y_val, y_val_pred):
#     # R^2 vs baseline that always predicts TRAIN mean
#     y_val_baseline = np.full_like(y_val, fill_value=y_train.mean(), dtype=float)
#     sse_model = np.sum((y_val - y_val_pred) ** 2)
#     sse_base  = np.sum((y_val - y_val_baseline) ** 2)
#     return 1 - sse_model / sse_base

# # List of candidate hyperparameter settings
# configs = [
#     {
#         "name": "simple_depth2",
#         "params": dict(
#             n_estimators=150,
#             max_depth=2,
#             learning_rate=0.05,
#             subsample=0.8,
#             colsample_bytree=0.8,
#             min_child_weight=10,
#             gamma=1.0,
#             reg_lambda=2.0,
#             reg_alpha=0.5,
#             objective="reg:squarederror",
#             random_state=42,
#             n_jobs=-1
#         )
#     },
#     {
#         "name": "medium_depth3",
#         "params": dict(
#             n_estimators=300,
#             max_depth=3,
#             learning_rate=0.05,
#             subsample=0.8,
#             colsample_bytree=0.8,
#             min_child_weight=5,
#             gamma=0.5,
#             reg_lambda=1.5,
#             reg_alpha=0.3,
#             objective="reg:squarederror",
#             random_state=42,
#             n_jobs=-1
#         )
#     },
#     {
#         "name": "slightly_richer",
#         "params": dict(
#             n_estimators=400,
#             max_depth=3,
#             learning_rate=0.05,
#             subsample=0.9,
#             colsample_bytree=0.9,
#             min_child_weight=3,
#             gamma=0.0,
#             reg_lambda=1.0,
#             reg_alpha=0.0,
#             objective="reg:squarederror",
#             random_state=42,
#             n_jobs=-1
#         )
#     },
# ]

# best_cfg = None
# best_val_rmse = float("inf")

# for cfg in configs:
#     print(f"\n=== Training config: {cfg['name']} ===")
#     model = XGBRegressor(**cfg["params"])
#     model.fit(X_train, y_train)

#     # Predictions
#     y_train_pred = model.predict(X_train)
#     y_val_pred   = model.predict(X_val)

#     # Standard metrics
#     tr_rmse, tr_mae, tr_r2 = regression_metrics(y_train, y_train_pred)
#     va_rmse, va_mae, va_r2 = regression_metrics(y_val,   y_val_pred)

#     # Time-series style R^2 vs train-mean baseline
#     ts_r2 = time_series_r2(y_train, y_val, y_val_pred)

#     print(f"Train: RMSE={tr_rmse:.4f}, MAE={tr_mae:.4f}, R^2={tr_r2:.4f}")
#     print(f"Val  : RMSE={va_rmse:.4f}, MAE={va_mae:.4f}, R^2={va_r2:.4f}")
#     print(f"Val  : Time-series R^2 vs train-mean baseline = {ts_r2:.4f}")

#     # Track best config by validation RMSE
#     if va_rmse < best_val_rmse:
#         best_val_rmse = va_rmse
#         best_cfg = {
#             "name": cfg["name"],
#             "model": model,
#             "val_rmse": va_rmse,
#             "val_mae": va_mae,
#             "val_r2": va_r2,
#             "ts_r2": ts_r2
#         }

# print("\n=== Best config by validation RMSE ===")
# print(best_cfg)

### A. Retrain on train + validation and re-check TEST

In [ ]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np


def regression_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = mse**0.5
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return rmse, mae, r2


def time_series_r2(y_train_all, y_test, y_test_pred):
    """R^2 vs baseline that always predicts TRAIN (train+val) mean."""
    y_test_baseline = np.full_like(y_test, fill_value=y_train_all.mean(), dtype=float)
    sse_model = np.sum((y_test - y_test_pred) ** 2)
    sse_base = np.sum((y_test - y_test_baseline) ** 2)
    return 1 - sse_model / sse_base


# 1) Combine TRAIN (2000–2009) + VAL (2010–2012) -> new training set
X_train_full = pd.concat([X_train, X_val], axis=0).reset_index(drop=True)
y_train_full = pd.concat([y_train, y_val], axis=0).reset_index(drop=True)

print("Train_full size (2000–2012):", len(y_train_full))
print("Test size (2013–2017):", len(y_test))

# 2) Retrain the tuned model (simple_depth2) on TRAIN+VAL
final_model_full = XGBRegressor(
    n_estimators=150,
    max_depth=2,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=10,
    gamma=1.0,
    reg_lambda=2.0,
    reg_alpha=0.5,
    objective="reg:squarederror",  # if this errors: "reg:linear"
    random_state=42,
    n_jobs=-1,
)

final_model_full.fit(X_train_full, y_train_full)

# 3) Baseline on TEST using TRAIN+VAL mean
train_full_mean = y_train_full.mean()
y_test_baseline = np.full_like(y_test, fill_value=train_full_mean, dtype=float)
base_rmse, base_mae, base_r2 = regression_metrics(y_test, y_test_baseline)
print("\n=== Baseline (predict train+val mean) on TEST (2013–2017) ===")
print(f"Baseline -> RMSE: {base_rmse:.4f}, MAE: {base_mae:.4f}, R^2: {base_r2:.4f}")

# 4) Model performance on TRAIN_FULL and TEST
y_train_full_pred = final_model_full.predict(X_train_full)
y_test_pred_full = final_model_full.predict(X_test)

tr_rmse, tr_mae, tr_r2 = regression_metrics(y_train_full, y_train_full_pred)
te_rmse, te_mae, te_r2 = regression_metrics(y_test, y_test_pred_full)
ts_r2_test = time_series_r2(y_train_full, y_test, y_test_pred_full)

print("\n=== Final XGBoost model retrained on 2000–2012 ===")
print(
    f"Train_full (2000–2012) -> RMSE: {tr_rmse:.4f}, MAE: {tr_mae:.4f}, R^2: {tr_r2:.4f}"
)
print(
    f"Test (2013–2017)       -> RMSE: {te_rmse:.4f}, MAE: {te_mae:.4f}, R^2: {te_r2:.4f}, "
    f"TS-R^2 vs train+val mean: {ts_r2_test:.4f}"
)

Train_full size (2000–2012): 12472
Test size (2013–2017): 5207

=== Baseline (predict train+val mean) on TEST (2013–2017) ===
Baseline -> RMSE: 3.5273, MAE: 2.5292, R^2: -0.2237

=== Final XGBoost model retrained on 2000–2012 ===
Train_full (2000–2012) -> RMSE: 3.3269, MAE: 2.1444, R^2: 0.7010
Test (2013–2017)       -> RMSE: 4.2288, MAE: 3.1373, R^2: -0.7588, TS-R^2 vs train+val mean: -0.4373


In [ ]:
df_all = df[df["yr"].between(2000, 2017)].copy()

df_all["period"] = pd.cut(
    df_all["yr"],
    bins=[1999, 2009, 2012, 2017],
    labels=["2000-2009", "2010-2012", "2013-2017"],
)

print(df_all.groupby("period")["hpi_yoy"].agg(["mean", "std", "min", "max"]))

               mean       std        min        max
period                                             
2000-2009  3.960190  5.971954 -38.965394  34.789236
2010-2012 -1.936060  3.307177 -16.864557  13.946363
2013-2017  3.887223  3.201101  -6.520841  23.814247


/var/folders/fh/f76kx5296m59wrx1dwbdp9q00000gn/T/ipykernel_14301/1733692543.py:9: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(df_all.groupby("period")["hpi_yoy"].agg(["mean", "std", "min", "max"]))


In [ ]:
test_df = df_model[df_model["yr"].between(2013, 2017)].copy()
test_df = test_df.assign(y_true=y_test.values, y_pred=final_model_full.predict(X_test))

print(test_df.groupby("yr")[["y_true", "y_pred"]].agg(["mean", "std"]))

        y_true              y_pred          
          mean       std      mean       std
yr                                          
2013  2.941200  3.977898  0.389964  0.998150
2014  3.266447  3.442881  0.874990  0.818060
2015  4.133609  2.534451  1.108006  0.820458
2016  4.297646  2.688386  1.267743  0.679597
2017  4.723083  2.707925  1.398850  0.623629
